Step 1 : Sentiment Analysis - Trained from Dataset

In [6]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping

# Step 1: Load the JSON dataset from file
with open('nlp_authentication_dataset.json', 'r') as file:
    data = json.load(file)

# Step 2: Extract texts and map sentiments to 1 (positive), 0 (negative), or skip neutral
texts = []
sentiments = []

for entry in data:
    sentiment = entry['sentiment'].lower()  # Convert to lowercase for consistency
    if sentiment == 'positive':
        sentiments.append(1)
    elif sentiment == 'negative':
        sentiments.append(0)
    else:
        continue  # Skip neutral or unrecognized sentiments
    texts.append(entry['raw_text'])  # Only append text if sentiment is valid

# Ensure there are valid texts and sentiments
if len(texts) == 0 or len(sentiments) == 0:
    raise ValueError("No valid data available for training after filtering.")

# Step 3: Tokenization and padding
tokenizer = Tokenizer(num_words=10000)  # Increase vocab size for better representation
tokenizer.fit_on_texts(texts)
X = tokenizer.texts_to_sequences(texts)
X = pad_sequences(X, padding='post', maxlen=200)  # Increase maxlen for longer texts

# Step 4: Convert sentiments to numpy arrays
y = np.array(sentiments)

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 6: Build a more complex LSTM model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=10000, output_dim=128, input_length=200),  # Larger embedding size
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True)),  # More units, return sequences for deeper layers
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),  # Additional LSTM layer
    tf.keras.layers.Dropout(0.5),  # Dropout for regularization
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Step 7: Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Step 8: Use Early Stopping to avoid overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Step 9: Train the model with more epochs and early stopping
model.fit(np.array(X_train), np.array(y_train), epochs=20, batch_size=32, validation_data=(np.array(X_test), np.array(y_test)), callbacks=[early_stopping])

# Step 10: Evaluate the model
loss, accuracy = model.evaluate(np.array(X_test), np.array(y_test))
print(f"Test Accuracy: {accuracy * 100:.2f}%")

# Step 11: Function to predict sentiment from new text input
def predict_sentiment(text):
    # Tokenize and pad the input text
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=200)  # Ensure the padding matches the training
    prediction = model.predict(padded)[0][0]
    if prediction >= 0.5:
        return "Positive"
    else:
        return "Negative"

# Step 12: Get user input and predict sentiment
while True:
    user_input = input("Enter a text to analyze sentiment (or type 'exit' to quit): ")
    if user_input.lower() == 'exit':
        break
    print("Sentiment:", predict_sentiment(user_input))


Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 335ms/step - accuracy: 0.5248 - loss: 0.6926 - val_accuracy: 0.5224 - val_loss: 0.6915
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 284ms/step - accuracy: 0.5081 - loss: 0.6957 - val_accuracy: 0.5224 - val_loss: 0.6907
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 544ms/step - accuracy: 0.5295 - loss: 0.6917 - val_accuracy: 0.5522 - val_loss: 0.6881
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 662ms/step - accuracy: 0.5292 - loss: 0.6914 - val_accuracy: 0.5299 - val_loss: 0.6871
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 655ms/step - accuracy: 0.5697 - loss: 0.6885 - val_accuracy: 0.5746 - val_loss: 0.6834
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 656ms/step - accuracy: 0.5449 - loss: 0.6871 - val_accuracy: 0.5299 - val_loss: 0.6821
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 635ms/step - accuracy: 0.5366 - loss: 0.6869 - val_accuracy: 0.5299 - val_loss: 0.6824
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 655ms/step - accuracy: 0.5474 - loss: 0.6875 - val_accura

Step 2: Stylometry Analysis - Trained from Dataset

In [15]:
import json
import numpy as np
import spacy

# Load Spacy's small model for NLP tasks
nlp = spacy.load("en_core_web_sm")

# Step 1: Load the JSON dataset from file
with open('nlp_authentication_dataset.json', 'r') as file:
    data = json.load(file)

# Step 2: Function to extract stylometric features from a given prompt
def extract_stylometry_features(prompt):
    doc = nlp(prompt)
    
    # Calculate POS tag distribution
    pos_counts = {'NN': 0, 'VB': 0, 'JJ': 0, 'RB': 0, 'PRP': 0}
    total_tokens = 0
    for token in doc:
        if token.pos_ in ['NOUN', 'VERB', 'ADJ', 'ADV', 'PRON']:
            total_tokens += 1
            if token.pos_ == 'NOUN':
                pos_counts['NN'] += 1
            elif token.pos_ == 'VERB':
                pos_counts['VB'] += 1
            elif token.pos_ == 'ADJ':
                pos_counts['JJ'] += 1
            elif token.pos_ == 'ADV':
                pos_counts['RB'] += 1
            elif token.pos_ == 'PRON':
                pos_counts['PRP'] += 1

    if total_tokens > 0:
        for pos in pos_counts:
            pos_counts[pos] = (pos_counts[pos] / total_tokens) * 100
    
    # Grammar usage (simplified: count sentences and subordinating conjunctions)
    sentences = list(doc.sents)
    correct_sentences = len(sentences)
    complex_sentences = sum(1 for token in doc if token.pos_ == 'SCONJ')  # Subordinating conjunctions

    # Average word length
    avg_word_len = np.mean([len(token.text) for token in doc if token.is_alpha])

    # Lexical diversity
    unique_words = len(set([token.text.lower() for token in doc if token.is_alpha]))
    total_words = len([token.text.lower() for token in doc if token.is_alpha])
    lexical_diversity = unique_words / total_words if total_words > 0 else 0

    # Readability score (Flesch-Kincaid, simplified calculation)
    total_words = len([token.text for token in doc if token.is_alpha])
    total_sentences = len(sentences)
    syllable_count = sum(len(token.text) // 3 for token in doc if token.is_alpha)
    readability_score = 206.835 - (1.015 * (total_words / total_sentences)) - (84.6 * (syllable_count / total_words)) if total_sentences > 0 and total_words > 0 else 0

    # Combine all features into one feature vector
    feature_vector = list(pos_counts.values()) + [correct_sentences, complex_sentences] + [avg_word_len, lexical_diversity, readability_score]
    
    return np.array(feature_vector)

# Step 3: Function to compare stylometry features with thresholds and print deviations
def compare_stylometry(user_id, prompt):
    # Find the stylometry features of the given user
    user_features_list = [entry['stylometry_features'] for entry in data if entry['user_id'] == user_id]
    
    if len(user_features_list) == 0:
        return "User not found."
    
    # Calculate the average stylometry features of the user
    avg_user_features = np.mean([
        list(features['pos_tag_distribution'].values()) +
        [features['grammar_usage']['correct_sentences'], features['grammar_usage']['complex_sentences']] +
        [features['average_word_length'], features['lexical_diversity'], features['readability_score']]
        for features in user_features_list
    ], axis=0)

    # Extract stylometry features from the given prompt
    prompt_features = extract_stylometry_features(prompt)

    # Step 4: Compare the extracted features with the user's average features
    pos_threshold = 10.0  # 10% for POS tag distribution
    numeric_threshold = 2.0  # +/- 2 for numeric values

    # Split the features into POS tag features (first 5) and numerical features (last 5)
    user_pos_features = avg_user_features[:5]
    user_numeric_features = avg_user_features[5:]
    
    prompt_pos_features = prompt_features[:5]
    prompt_numeric_features = prompt_features[5:]

    # Print the stylometry features of the user and the prompt
    print("\nStylometry Comparison:")
    print(f"{'Feature':<25}{'User Feature':<20}{'Prompt Feature':<20}{'Deviation'}")
    print("-" * 80)

    # Check POS tag distribution (percentage comparison)
    pos_match = True
    for i, (user_pos, prompt_pos) in enumerate(zip(user_pos_features, prompt_pos_features)):
        deviation = abs(user_pos - prompt_pos)
        if deviation > pos_threshold:
            pos_match = False
        print(f"POS Tag {i + 1:<19} {user_pos:<20} {prompt_pos:<20} {deviation:.2f}%")

    # Check numerical features (number comparison)
    num_match = True
    for i, (user_num, prompt_num) in enumerate(zip(user_numeric_features, prompt_numeric_features)):
        deviation = abs(user_num - prompt_num)
        if deviation > numeric_threshold:
            num_match = False
        print(f"Numeric Feature {i + 1:<15} {user_num:<20} {prompt_num:<20} {deviation:.2f}")

    # Check if both POS and numeric features are within thresholds
    if pos_match and num_match:
        print("\nResult: Stylometry match.")
        return "Stylometry match."
    else:
        print("\nResult: Stylometry does not match.")
        return "Stylometry does not match."

# Step 5: Test the system by providing a user ID and a prompt
user_id_input = input("Enter user ID: ")
prompt_input = input("Enter the text prompt: ")

result = compare_stylometry(user_id_input, prompt_input)



Stylometry Comparison:
Feature                  User Feature        Prompt Feature      Deviation
--------------------------------------------------------------------------------
POS Tag 1                   15.969999999999999   42.857142857142854   26.89%
POS Tag 2                   20.089999999999996   14.285714285714285   5.80%
POS Tag 3                   18.995               28.57142857142857    9.58%
POS Tag 4                   18.56                0.0                  18.56%
POS Tag 5                   19.0                 14.285714285714285   4.71%
Numeric Feature 1               2.75                 1.0                  1.75
Numeric Feature 2               1.15                 0.0                  1.15
Numeric Feature 3               5.6845               3.75                 1.93
Numeric Feature 4               0.66                 1.0                  0.34
Numeric Feature 5               7.584999999999999    124.155              116.57

Result: Stylometry does not match.


In [4]:
import json
import numpy as np
import spacy

# Load Spacy's small model for NLP tasks
nlp = spacy.load("en_core_web_sm")

# Step 1: Load the JSON dataset and extract stylometry features
def load_dataset(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

# Step 2: Function to extract stylometry features from a prompt
def extract_stylometry_features(prompt):
    doc = nlp(prompt)
    
    # Calculate POS tag distribution (Nouns, Verbs, Adjectives, Adverbs, Pronouns)
    pos_counts = {'NN': 0, 'VB': 0, 'JJ': 0, 'RB': 0, 'PRP': 0}
    total_tokens = 0
    for token in doc:
        if token.pos_ in ['NOUN', 'VERB', 'ADJ', 'ADV', 'PRON']:
            total_tokens += 1
            if token.pos_ == 'NOUN':
                pos_counts['NN'] += 1
            elif token.pos_ == 'VERB':
                pos_counts['VB'] += 1
            elif token.pos_ == 'ADJ':
                pos_counts['JJ'] += 1
            elif token.pos_ == 'ADV':
                pos_counts['RB'] += 1
            elif token.pos_ == 'PRON':
                pos_counts['PRP'] += 1

    if total_tokens > 0:
        for pos in pos_counts:
            pos_counts[pos] = (pos_counts[pos] / total_tokens) * 100
    
    # Grammar usage (correct sentences, complex sentences = subordinating conjunctions)
    sentences = list(doc.sents)
    correct_sentences = len(sentences)
    complex_sentences = sum(1 for token in doc if token.pos_ == 'SCONJ')  # Subordinating conjunctions

    # Average word length
    avg_word_len = np.mean([len(token.text) for token in doc if token.is_alpha])

    # Lexical diversity
    unique_words = len(set([token.text.lower() for token in doc if token.is_alpha]))
    total_words = len([token.text.lower() for token in doc if token.is_alpha])
    lexical_diversity = unique_words / total_words if total_words > 0 else 0

    # Readability score (Flesch-Kincaid simplified formula)
    total_words = len([token.text for token in doc if token.is_alpha])
    total_sentences = len(sentences)
    syllable_count = sum(len(token.text) // 3 for token in doc if token.is_alpha)
    readability_score = 206.835 - (1.015 * (total_words / total_sentences)) - (84.6 * (syllable_count / total_words)) if total_sentences > 0 and total_words > 0 else 0

    # Combine all features into a vector
    feature_vector = list(pos_counts.values()) + [correct_sentences, complex_sentences, avg_word_len, lexical_diversity, readability_score]
    
    return np.array(feature_vector)

# Step 3: Compute the average statistics for each user from the dataset
def calculate_user_statistics(data):
    user_statistics = {}
    
    for entry in data:
        user_id = entry['user_id']
        features = entry['stylometry_features']
        
        pos_tags = list(features['pos_tag_distribution'].values())
        grammar_usage = [features['grammar_usage']['correct_sentences'], features['grammar_usage']['complex_sentences']]
        avg_word_len = [features['average_word_length']]
        lexical_diversity = [features['lexical_diversity']]
        readability = [features['readability_score']]

        # Combine all features
        feature_vector = pos_tags + grammar_usage + avg_word_len + lexical_diversity + readability
        
        # If user already exists, append the new vector to their data
        if user_id in user_statistics:
            user_statistics[user_id].append(np.array(feature_vector))
        else:
            user_statistics[user_id] = [np.array(feature_vector)]
    
    # Calculate the average statistics for each user
    for user_id in user_statistics:
        user_statistics[user_id] = np.mean(user_statistics[user_id], axis=0)
    
    return user_statistics

# Step 4: Function to compare the prompt's features with the user's stored statistics and print deviations
def compare_stylometry(user_stats, prompt_features):
    pos_threshold = 10.0  # 10% for POS tags
    numeric_threshold = 2.0  # +/- 2 for numeric values
    
    # Split the features into POS tag and numeric features
    user_pos_features = user_stats[:5]
    user_numeric_features = user_stats[5:]
    
    prompt_pos_features = prompt_features[:5]
    prompt_numeric_features = prompt_features[5:]

    # Print the stylometry comparison details
    print("\nStylometry Comparison:")
    print(f"{'Feature':<25}{'User Statistic':<20}{'Prompt Value':<20}{'Deviation'}")
    print("-" * 80)

    # Compare POS tag percentages
    pos_match = True
    for i, (user_pos, prompt_pos) in enumerate(zip(user_pos_features, prompt_pos_features)):
        deviation = abs(user_pos - prompt_pos)
        within_range = deviation <= pos_threshold
        pos_match = pos_match and within_range
        print(f"POS Tag {i + 1:<19} {user_pos:<20.2f} {prompt_pos:<20.2f} {deviation:.2f}% {'(OK)' if within_range else '(NOT OK)'}")

    # Compare numeric values
    num_match = True
    for i, (user_num, prompt_num) in enumerate(zip(user_numeric_features, prompt_numeric_features)):
        deviation = abs(user_num - prompt_num)
        within_range = deviation <= numeric_threshold
        num_match = num_match and within_range
        print(f"Numeric Feature {i + 1:<15} {user_num:<20.2f} {prompt_num:<20.2f} {deviation:.2f} {'(OK)' if within_range else '(NOT OK)'}")

    # Return final result
    if pos_match and num_match:
        print("\nResult: Authentication successful. Stylometry matches.")
    else:
        print("\nResult: Authentication failed. Stylometry does not match.")
    
    return pos_match and num_match

# Step 5: Main function to run the system
def main():
    # Load the dataset
    dataset_path = 'nlp_authentication_dataset.json'  # Update with actual path
    data = load_dataset(dataset_path)

    # Calculate user statistics
    user_statistics = calculate_user_statistics(data)

    # Step 6: Prompt the user for a user ID and a prompt
    user_id = input("Enter user ID: ")
    
    if user_id not in user_statistics:
        print("User ID not found.")
        return

    prompt = input("Enter a text prompt: ")
    
    # Step 7: Extract stylometry features from the prompt
    prompt_features = extract_stylometry_features(prompt)

    # Step 8: Compare prompt features with user's stored statistics
    user_stats = user_statistics[user_id]
    is_authenticated = compare_stylometry(user_stats, prompt_features)

# Run the main function
if __name__ == "__main__":
    main()



Stylometry Comparison:
Feature                  User Statistic      Prompt Value        Deviation
--------------------------------------------------------------------------------
POS Tag 1                   15.97                52.94                36.97% (NOT OK)
POS Tag 2                   20.09                29.41                9.32% (OK)
POS Tag 3                   19.00                11.76                7.23% (OK)
POS Tag 4                   18.56                5.88                 12.68% (NOT OK)
POS Tag 5                   19.00                0.00                 19.00% (NOT OK)
Numeric Feature 1               2.75                 4.00                 1.25 (OK)
Numeric Feature 2               1.15                 0.00                 1.15 (OK)
Numeric Feature 3               5.68                 3.82                 1.87 (OK)
Numeric Feature 4               0.66                 0.70                 0.04 (OK)
Numeric Feature 5               7.58                 111.30     

Prompt length checker 

In [9]:
import json

# Step 1: Load the dataset from file
def load_dataset(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

# Step 2: Calculate average prompt length based on the 'prompt_length' field for each user
def calculate_average_prompt_length(data):
    user_prompt_lengths = {}
    
    # Iterate through the dataset to accumulate the 'prompt_length' for each user
    for entry in data:
        user_id = entry['user_id']
        prompt_length = entry['prompt_length']  # Use the 'prompt_length' field from the dataset
        
        # Add the prompt length to the user's list of prompt lengths
        if user_id in user_prompt_lengths:
            user_prompt_lengths[user_id].append(prompt_length)
        else:
            user_prompt_lengths[user_id] = [prompt_length]
    
    # Calculate the average prompt length for each user
    user_average_lengths = {user_id: sum(lengths) / len(lengths) for user_id, lengths in user_prompt_lengths.items()}
    
    return user_average_lengths

# Step 3: Prompt for user input and authenticate based on character count
def authenticate_user(user_average_lengths, tolerance=15):  # Default tolerance set to 50 characters
    user_id = input("Enter user ID: ")
    
    # Check if the user ID exists in the dataset
    if user_id not in user_average_lengths:
        print("User ID not found.")
        return
    
    # Get the user's average prompt length in characters from the 'prompt_length' field
    average_length = user_average_lengths[user_id]
    
    # Print the average prompt length for the user
    print(f"Average prompt length for user {user_id}: {average_length:.2f} characters.")
    
    # Prompt the user for a new input prompt
    prompt = input("Enter a text prompt: ")
    prompt_length = len(prompt)  # Calculate the number of characters in the new prompt
    
    # Check if the prompt length is within the acceptable range (±tolerance)
    if average_length - tolerance <= prompt_length <= average_length + tolerance:
        print("Authentication successful.")
    else:
        print(f"Authentication failed. Prompt length: {prompt_length}, Expected: {average_length:.2f} ± {tolerance} characters.")

# Main function to load dataset, calculate averages, and authenticate user
def main():
    # Load the dataset (replace with actual dataset path)
    dataset_path = 'nlp_authentication_dataset.json'
    data = load_dataset(dataset_path)
    
    # Calculate average prompt lengths (in characters) for each user using 'prompt_length' field
    user_average_lengths = calculate_average_prompt_length(data)
    
    # Authenticate the user based on character count
    authenticate_user(user_average_lengths)

# Run the script
if __name__ == "__main__":
    main()


Average prompt length for user User_001: 65.15 characters.
Authentication failed. Prompt length: 27, Expected: 65.15 ± 15 characters.


Prompt length checker with mean deviation for each user as tolerance.

In [12]:
import json

# Step 1: Load the dataset from file
def load_dataset(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

# Step 2: Calculate average prompt length and mean deviation for each user
def calculate_average_and_deviation(data):
    user_prompt_lengths = {}
    
    # Iterate through the dataset to accumulate the 'prompt_length' for each user
    for entry in data:
        user_id = entry['user_id']
        prompt_length = entry['prompt_length']  # Use the 'prompt_length' field from the dataset
        
        # Add the prompt length to the user's list of prompt lengths
        if user_id in user_prompt_lengths:
            user_prompt_lengths[user_id].append(prompt_length)
        else:
            user_prompt_lengths[user_id] = [prompt_length]
    
    # Calculate the average prompt length and mean deviation for each user
    user_average_lengths = {}
    user_mean_deviation = {}

    for user_id, lengths in user_prompt_lengths.items():
        # Calculate average prompt length
        average_length = sum(lengths) / len(lengths)
        user_average_lengths[user_id] = average_length
        
        # Calculate mean deviation (absolute deviation from the average length)
        deviations = [abs(length - average_length) for length in lengths]
        mean_deviation = sum(deviations) / len(deviations)
        user_mean_deviation[user_id] = mean_deviation
    
    return user_average_lengths, user_mean_deviation

# Step 3: Prompt for user input and authenticate based on character count
def authenticate_user(user_average_lengths, user_mean_deviation):
    user_id = input("Enter user ID: ")
    
    # Check if the user ID exists in the dataset
    if user_id not in user_average_lengths:
        print("User ID not found.")
        return
    
    # Get the user's average prompt length and mean deviation (tolerance)
    average_length = user_average_lengths[user_id]
    tolerance = user_mean_deviation[user_id]
    
    # Print the average prompt length and tolerance for the user
    print(f"Average prompt length for user {user_id}: {average_length:.2f} characters.")
    print(f"Using a tolerance of ±{tolerance:.2f} characters for authentication.")
    
    # Prompt the user for a new input prompt
    prompt = input("Enter a text prompt: ")
    prompt_length = len(prompt)  # Calculate the number of characters in the new prompt
    
    # Check if the prompt length is within the acceptable range (±tolerance)
    if average_length - tolerance <= prompt_length <= average_length + tolerance:
        print("Authentication successful.")
    else:
        print(f"Authentication failed. Prompt length: {prompt_length}, Expected: {average_length:.2f} ± {tolerance:.2f} characters.")

# Main function to load dataset, calculate averages, and authenticate user
def main():
    # Load the dataset (replace with actual dataset path)
    dataset_path = 'nlp_authentication_dataset.json'
    data = load_dataset(dataset_path)
    
    # Calculate average prompt lengths (in characters) and mean deviation for each user
    user_average_lengths, user_mean_deviation = calculate_average_and_deviation(data)
    
    # Authenticate the user based on character count and user-specific tolerance
    authenticate_user(user_average_lengths, user_mean_deviation)

# Run the script
if __name__ == "__main__":
    main()


Average prompt length for user User_003: 65.65 characters.
Using a tolerance of ±8.32 characters for authentication.
Authentication failed. Prompt length: 27, Expected: 65.65 ± 8.32 characters.


Time and Location checker

In [14]:
# ! pip install geocoder

In [15]:
import json
from collections import Counter
from datetime import datetime
import geocoder

# Step 1: Load the dataset from file
def load_dataset(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

# Step 2: Calculate the most frequent time_of_day and location for each user
def calculate_frequent_time_location(data):
    user_time_location = {}
    
    # Iterate through the dataset to accumulate time_of_day and location for each user
    for entry in data:
        user_id = entry['user_id']
        time_of_day = entry['time_of_day']  # Get the time_of_day field from the dataset
        location = entry['location']  # Get the location field from the dataset
        
        # Initialize dictionary for user if not already done
        if user_id not in user_time_location:
            user_time_location[user_id] = {
                'time_of_day': [],
                'location': []
            }
        
        # Append the time_of_day and location values
        user_time_location[user_id]['time_of_day'].append(time_of_day)
        user_time_location[user_id]['location'].append(location)
    
    # Calculate the most frequent time_of_day and location for each user
    user_most_frequent = {}
    
    for user_id, values in user_time_location.items():
        # Find the most frequent time_of_day and location
        most_frequent_time = Counter(values['time_of_day']).most_common(1)[0][0]  # Get the most frequent time_of_day
        most_frequent_location = Counter(values['location']).most_common(1)[0][0]  # Get the most frequent location
        
        # Store the results for the user
        user_most_frequent[user_id] = {
            'most_frequent_time': most_frequent_time,
            'most_frequent_location': most_frequent_location
        }
    
    return user_most_frequent

# Step 3: Automatically detect time of day based on system time
def get_time_of_day():
    current_hour = datetime.now().hour
    if 6 <= current_hour < 12:
        return 'morning'
    elif 12 <= current_hour < 18:
        return 'afternoon'
    elif 18 <= current_hour < 22:
        return 'evening'
    else:
        return 'night'

# Step 4: Automatically detect location using geocoder
def get_location():
    g = geocoder.ip('me')
    return g.city  # Returns the city name based on IP address

# Step 5: Prompt for user input and authenticate based on detected time_of_day and location
def authenticate_user(user_most_frequent):
    user_id = input("Enter user ID: ")
    
    # Check if the user ID exists in the dataset
    if user_id not in user_most_frequent:
        print("User ID not found.")
        return
    
    # Get the user's most frequent time_of_day and location
    most_frequent_time = user_most_frequent[user_id]['most_frequent_time']
    most_frequent_location = user_most_frequent[user_id]['most_frequent_location']
    
    # Print the most frequent time_of_day and location for the user
    print(f"Most frequent time of day for user {user_id}: {most_frequent_time}")
    print(f"Most frequent location for user {user_id}: {most_frequent_location}")
    
    # Automatically detect the current time of day and location
    detected_time_of_day = get_time_of_day()
    detected_location = get_location()
    
    print(f"Detected time of day: {detected_time_of_day}")
    print(f"Detected location: {detected_location}")
    
    # Authenticate only if both time_of_day and location match the user's most frequent values
    if most_frequent_time == detected_time_of_day and most_frequent_location == detected_location:
        print("Authentication successful.")
    else:
        print("Authentication failed. The detected time of day or location does not match.")

# Main function to load dataset, calculate frequent values, and authenticate user
def main():
    # Load the dataset (replace with actual dataset path)
    dataset_path = 'nlp_authentication_dataset.json'
    data = load_dataset(dataset_path)
    
    # Calculate most frequent time_of_day and location for each user
    user_most_frequent = calculate_frequent_time_location(data)
    
    # Authenticate the user based on time_of_day and location
    authenticate_user(user_most_frequent)

# Run the script
if __name__ == "__main__":
    main()


Most frequent time of day for user User_001: morning
Most frequent location for user User_001: Europe/London
Detected time of day: afternoon
Detected location: Thessaloníki
Authentication failed. The detected time of day or location does not match.


SENTIMENT ANALYSIS

In [17]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Step 1: Load the dataset
def load_dataset(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

# Step 2: Preprocess the data
def preprocess_data(data, max_num_words=5000, max_seq_length=100):
    texts = []
    sentiments = []

    for entry in data:
        sentiment = entry['sentiment'].lower()  # assuming sentiment is 'positive', 'negative', or 'neutral'
        if sentiment == 'positive':
            sentiments.append(1)
        elif sentiment == 'negative':
            sentiments.append(0)
        else:
            continue  # we ignore neutral sentiments
        texts.append(entry['raw_text'])  # raw_text contains the user input

    tokenizer = Tokenizer(num_words=max_num_words)
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    data = pad_sequences(sequences, maxlen=max_seq_length)

    return data, np.array(sentiments), tokenizer

# Step 3: Build and train the LSTM model
def build_and_train_model(X_train, y_train, X_test, y_test, max_num_words=5000, max_seq_length=100):
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(input_dim=max_num_words, output_dim=128, input_length=max_seq_length),
        tf.keras.layers.LSTM(128, return_sequences=False),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

    return model

# Step 4: Save the trained model using the Keras format
def save_model(model, tokenizer, model_path='sentiment_model.keras', tokenizer_path='tokenizer.json'):
    # Save the model in the Keras format
    model.save(model_path)  # Uses .keras format instead of HDF5 (.h5)
    
    # Save the tokenizer as well
    with open(tokenizer_path, 'w') as f:
        json.dump(tokenizer.to_json(), f)

# Main function for training
def main():
    # Load the dataset (replace with actual path)
    dataset_path = 'nlp_authentication_dataset.json'
    data = load_dataset(dataset_path)

    # Preprocess the data
    X, y, tokenizer = preprocess_data(data)

    # Split into training and test data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Build and train the model
    model = build_and_train_model(X_train, y_train, X_test, y_test)

    # Save the trained model and tokenizer
    save_model(model, tokenizer)

if __name__ == "__main__":
    main()


Epoch 1/5
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 125ms/step - accuracy: 0.4821 - loss: 0.6940 - val_accuracy: 0.5224 - val_loss: 0.6910
Epoch 2/5
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 74ms/step - accuracy: 0.5252 - loss: 0.6901 - val_accuracy: 0.5522 - val_loss: 0.6883
Epoch 3/5
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - accuracy: 0.5544 - loss: 0.6863 - val_accuracy: 0.5746 - val_loss: 0.6856
Epoch 4/5
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step - accuracy: 0.5354 - loss: 0.6873 - val_accuracy: 0.5075 - val_loss: 0.6826
Epoch 5/5
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step - accuracy: 0.5262 - loss: 0.6933 - val_accuracy: 0.5299 - val_loss: 0.6836


In [19]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from collections import Counter

# Step 1: Load the trained model and tokenizer
def load_model_and_tokenizer(model_path='sentiment_model.keras', tokenizer_path='tokenizer.json'):
    model = tf.keras.models.load_model(model_path)
    
    with open(tokenizer_path, 'r') as f:
        tokenizer_json = json.load(f)
        tokenizer = tf.keras.preprocessing.text.tokenizer_from_json(tokenizer_json)
    
    return model, tokenizer

# Step 2: Calculate the two most frequent sentiments for each user
def calculate_frequent_sentiments(data):
    user_sentiments = {}
    
    for entry in data:
        user_id = entry['user_id']
        sentiment = entry['sentiment'].lower()
        if sentiment in ['positive', 'negative']:
            if user_id not in user_sentiments:
                user_sentiments[user_id] = []
            user_sentiments[user_id].append(sentiment)
    
    user_top_sentiments = {}
    for user_id, sentiments in user_sentiments.items():
        sentiment_counts = Counter(sentiments)
        top_2 = [s for s, _ in sentiment_counts.most_common(2)]
        user_top_sentiments[user_id] = top_2
    
    return user_top_sentiments

# Step 3: Predict sentiment of the input
def predict_sentiment(model, tokenizer, text, max_seq_length=100):
    sequences = tokenizer.texts_to_sequences([text])
    padded_sequences = pad_sequences(sequences, maxlen=max_seq_length)
    prediction = model.predict(padded_sequences)[0][0]
    
    if prediction >= 0.5:
        return 'positive'
    else:
        return 'negative'

# Step 4: Authenticate user based on sentiment
def authenticate_user(model, tokenizer, user_top_sentiments):
    user_id = input("Enter user ID: ")
    
    if user_id not in user_top_sentiments:
        print("User ID not found.")
        return
    
    # Get user's top 2 frequent sentiments
    top_2_sentiments = user_top_sentiments[user_id]
    print(f"User's top 2 frequent sentiments: {top_2_sentiments}")
    
    # Prompt for user input
    input_text = input("Enter a text prompt: ")
    
    # Predict sentiment of the input
    predicted_sentiment = predict_sentiment(model, tokenizer, input_text)
    print(f"Predicted sentiment: {predicted_sentiment}")
    
    # Authenticate if sentiment matches one of the top 2
    if predicted_sentiment in top_2_sentiments:
        print("Authentication successful.")
    else:
        print("Authentication failed.")

# Main function for loading, testing, and authentication
def main():
    # Load the dataset and calculate top sentiments (replace with actual dataset path)
    dataset_path = 'nlp_authentication_dataset.json'
    with open(dataset_path, 'r') as file:
        data = json.load(file)

    # Calculate the top 2 frequent sentiments for each user
    user_top_sentiments = calculate_frequent_sentiments(data)

    # Load the trained model and tokenizer
    model, tokenizer = load_model_and_tokenizer()

    # Authenticate user based on sentiment
    authenticate_user(model, tokenizer, user_top_sentiments)

if __name__ == "__main__":
    main()


User's top 2 frequent sentiments: ['negative', 'positive']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 229ms/step
Predicted sentiment: negative
Authentication successful.


SENTIMENT ANALYSIS - IMPROVED ACCURACY

In [20]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping

# Step 1: Load the dataset
def load_dataset(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

# Step 2: Preprocess the data for multi-class classification
def preprocess_data(data, max_num_words=5000, max_seq_length=100):
    texts = []
    sentiments = []

    for entry in data:
        sentiment = entry['sentiment'].lower()
        if sentiment == 'positive':
            sentiments.append(2)  # Label positive as 2
        elif sentiment == 'very positive':
            sentiments.append(3)  # Label very positive as 3
        elif sentiment == 'negative':
            sentiments.append(1)  # Label negative as 1
        elif sentiment == 'very negative':
            sentiments.append(0)  # Label very negative as 0
        elif sentiment == 'neutral':
            sentiments.append(4)  # Label neutral as 4
        else:
            continue  # Ignore any other unknown sentiments
        texts.append(entry['raw_text'])

    tokenizer = Tokenizer(num_words=max_num_words)
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    data = pad_sequences(sequences, maxlen=max_seq_length)

    return data, np.array(sentiments), tokenizer

# Step 3: Build and train the LSTM model for multi-class classification
def build_and_train_model(X_train, y_train, X_test, y_test, max_num_words=5000, max_seq_length=100, epochs=10):
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(input_dim=max_num_words, output_dim=128, input_length=max_seq_length),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=False)),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(5, activation='softmax')  # Softmax for multi-class classification (5 categories)
    ])

    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    # Adding early stopping to prevent overfitting
    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    model.fit(X_train, y_train, epochs=epochs, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stopping])

    return model

# Step 4: Save the trained model using the Keras format
def save_model(model, tokenizer, model_path='sentiment_model.keras', tokenizer_path='tokenizer.json'):
    # Save the model in the Keras format
    model.save(model_path)  # Uses .keras format instead of HDF5 (.h5)
    
    # Save the tokenizer as well
    with open(tokenizer_path, 'w') as f:
        json.dump(tokenizer.to_json(), f)

# Main function for training
def main():
    # Load the dataset (replace with actual path)
    dataset_path = 'nlp_authentication_dataset.json'
    data = load_dataset(dataset_path)

    # Preprocess the data
    X, y, tokenizer = preprocess_data(data)

    # Split into training and test data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Build and train the model
    model = build_and_train_model(X_train, y_train, X_test, y_test, epochs=10)

    # Save the trained model and tokenizer
    save_model(model, tokenizer)

if __name__ == "__main__":
    main()


Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 113ms/step - accuracy: 0.2852 - loss: 1.3782 - val_accuracy: 0.3000 - val_loss: 1.1479
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 82ms/step - accuracy: 0.3575 - loss: 1.1479 - val_accuracy: 0.3700 - val_loss: 1.1083
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 105ms/step - accuracy: 0.3589 - loss: 1.1361 - val_accuracy: 0.3750 - val_loss: 1.1097
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 99ms/step - accuracy: 0.3525 - loss: 1.1155 - val_accuracy: 0.3300 - val_loss: 1.1077
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 95ms/step - accuracy: 0.3520 - loss: 1.1104 - val_accuracy: 0.3700 - val_loss: 1.1005
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 94ms/step - accuracy: 0.3545 - loss: 1.1068 - val_accuracy: 0.3700 - val_loss: 1.1057
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - accuracy: 0.3595 - loss: 1.1142 - val_accuracy: 0.3050 - val_loss: 1.1231
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 95ms/step - accuracy: 0.3523 - loss: 1.1067 - val_accuracy: 0.3500 -

In [21]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from collections import Counter

# Step 1: Load the trained model and tokenizer
def load_model_and_tokenizer(model_path='sentiment_model.keras', tokenizer_path='tokenizer.json'):
    model = tf.keras.models.load_model(model_path)
    
    with open(tokenizer_path, 'r') as f:
        tokenizer_json = json.load(f)
        tokenizer = tf.keras.preprocessing.text.tokenizer_from_json(tokenizer_json)
    
    return model, tokenizer

# Step 2: Calculate the two most frequent sentiments for each user
def calculate_frequent_sentiments(data):
    user_sentiments = {}
    
    for entry in data:
        user_id = entry['user_id']
        sentiment = entry['sentiment'].lower()
        if sentiment in ['positive', 'negative', 'very positive', 'very negative', 'neutral']:
            if user_id not in user_sentiments:
                user_sentiments[user_id] = []
            user_sentiments[user_id].append(sentiment)
    
    user_top_sentiments = {}
    for user_id, sentiments in user_sentiments.items():
        sentiment_counts = Counter(sentiments)
        top_2 = [s for s, _ in sentiment_counts.most_common(2)]
        user_top_sentiments[user_id] = top_2
    
    return user_top_sentiments

# Step 3: Predict sentiment of the input
def predict_sentiment(model, tokenizer, text, max_seq_length=100):
    sequences = tokenizer.texts_to_sequences([text])
    padded_sequences = pad_sequences(sequences, maxlen=max_seq_length)
    prediction = np.argmax(model.predict(padded_sequences), axis=1)[0]
    
    # Map numeric prediction to sentiment category
    sentiment_mapping = {0: 'very negative', 1: 'negative', 2: 'positive', 3: 'very positive', 4: 'neutral'}
    return sentiment_mapping[prediction]

# Step 4: Authenticate user based on sentiment
def authenticate_user(model, tokenizer, user_top_sentiments):
    user_id = input("Enter user ID: ")
    
    if user_id not in user_top_sentiments:
        print("User ID not found.")
        return
    
    # Get user's top 2 frequent sentiments
    top_2_sentiments = user_top_sentiments[user_id]
    print(f"User's top 2 frequent sentiments: {top_2_sentiments}")
    
    # Prompt for user input
    input_text = input("Enter a text prompt: ")
    
    # Predict sentiment of the input
    predicted_sentiment = predict_sentiment(model, tokenizer, input_text)
    print(f"Predicted sentiment: {predicted_sentiment}")
    
    # Authenticate if sentiment matches one of the top 2
    if predicted_sentiment in top_2_sentiments:
        print("Authentication successful.")
    else:
        print("Authentication failed.")

# Main function for loading, testing, and authentication
def main():
    # Load the dataset and calculate top sentiments (replace with actual dataset path)
    dataset_path = 'nlp_authentication_dataset.json'
    with open(dataset_path, 'r') as file:
        data = json.load(file)

    # Calculate the top 2 frequent sentiments for each user
    user_top_sentiments = calculate_frequent_sentiments(data)

    # Load the trained model and tokenizer
    model, tokenizer = load_model_and_tokenizer()

    # Authenticate user based on sentiment
    authenticate_user(model, tokenizer, user_top_sentiments)

if __name__ == "__main__":
    main()


User's top 2 frequent sentiments: ['negative', 'positive']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step
Predicted sentiment: negative
Authentication successful.


Sentiment analysis - modified dataset with 5 categories

In [23]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping

# Step 1: Load the dataset
def load_dataset(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

# Step 2: Preprocess the data for multi-class classification
def preprocess_data(data, max_num_words=5000, max_seq_length=100):
    texts = []
    sentiments = []

    for entry in data:
        sentiment = entry['sentiment'].lower()
        if sentiment == 'positive':
            sentiments.append(2)  # Label positive as 2
        elif sentiment == 'very positive':
            sentiments.append(3)  # Label very positive as 3
        elif sentiment == 'negative':
            sentiments.append(1)  # Label negative as 1
        elif sentiment == 'very negative':
            sentiments.append(0)  # Label very negative as 0
        elif sentiment == 'neutral':
            sentiments.append(4)  # Label neutral as 4
        else:
            continue  # Ignore any other unknown sentiments
        texts.append(entry['raw_text'])

    tokenizer = Tokenizer(num_words=max_num_words)
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    data = pad_sequences(sequences, maxlen=max_seq_length)

    return data, np.array(sentiments), tokenizer

# Step 3: Build and train the LSTM model for multi-class classification
def build_and_train_model(X_train, y_train, X_test, y_test, max_num_words=5000, max_seq_length=100, epochs=10):
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(input_dim=max_num_words, output_dim=128, input_length=max_seq_length),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=False)),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(5, activation='softmax')  # Softmax for multi-class classification (5 categories)
    ])

    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    # Adding early stopping to prevent overfitting
    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    model.fit(X_train, y_train, epochs=epochs, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stopping])

    return model

# Step 4: Save the trained model using the Keras format
def save_model(model, tokenizer, model_path='sentiment_model.keras', tokenizer_path='tokenizer.json'):
    # Save the model in the Keras format
    model.save(model_path)  # Uses .keras format instead of HDF5 (.h5)
    
    # Save the tokenizer as well
    with open(tokenizer_path, 'w') as f:
        json.dump(tokenizer.to_json(), f)

# Main function for training
def main():
    # Load the dataset (replace with actual path)
    dataset_path = 'modified_nlp_authentication_dataset.json'
    data = load_dataset(dataset_path)

    # Preprocess the data
    X, y, tokenizer = preprocess_data(data)

    # Split into training and test data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Build and train the model
    model = build_and_train_model(X_train, y_train, X_test, y_test, epochs=10)

    # Save the trained model and tokenizer
    save_model(model, tokenizer)

if __name__ == "__main__":
    main()


Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.3428 - loss: 1.3806 - val_accuracy: 0.3300 - val_loss: 1.1642
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 0.3000 - loss: 1.1901 - val_accuracy: 0.3100 - val_loss: 1.1096
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.3335 - loss: 1.1228 - val_accuracy: 0.3300 - val_loss: 1.1136
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 167ms/step - accuracy: 0.3281 - loss: 1.1419 - val_accuracy: 0.3200 - val_loss: 1.1107
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 176ms/step - accuracy: 0.3846 - loss: 1.0957 - val_accuracy: 0.3250 - val_loss: 1.1016
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 159ms/step - accuracy: 0.3427 - loss: 1.1202 - val_accuracy: 0.3350 - val_loss: 1.1036
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 162ms/step - accuracy: 0.3326 - loss: 1.1169 - val_accuracy: 0.3500 - val_loss: 1.1044
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 166ms/step - accuracy: 0.3430 - loss: 1.1223 - val_accuracy: 0.375

In [24]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from collections import Counter

# Step 1: Load the trained model and tokenizer
def load_model_and_tokenizer(model_path='sentiment_model.keras', tokenizer_path='tokenizer.json'):
    model = tf.keras.models.load_model(model_path)
    
    with open(tokenizer_path, 'r') as f:
        tokenizer_json = json.load(f)
        tokenizer = tf.keras.preprocessing.text.tokenizer_from_json(tokenizer_json)
    
    return model, tokenizer

# Step 2: Calculate the two most frequent sentiments for each user
def calculate_frequent_sentiments(data):
    user_sentiments = {}
    
    for entry in data:
        user_id = entry['user_id']
        sentiment = entry['sentiment'].lower()
        if sentiment in ['positive', 'negative', 'very positive', 'very negative', 'neutral']:
            if user_id not in user_sentiments:
                user_sentiments[user_id] = []
            user_sentiments[user_id].append(sentiment)
    
    user_top_sentiments = {}
    for user_id, sentiments in user_sentiments.items():
        sentiment_counts = Counter(sentiments)
        top_2 = [s for s, _ in sentiment_counts.most_common(2)]
        user_top_sentiments[user_id] = top_2
    
    return user_top_sentiments

# Step 3: Predict sentiment of the input
def predict_sentiment(model, tokenizer, text, max_seq_length=100):
    sequences = tokenizer.texts_to_sequences([text])
    padded_sequences = pad_sequences(sequences, maxlen=max_seq_length)
    prediction = np.argmax(model.predict(padded_sequences), axis=1)[0]
    
    # Map numeric prediction to sentiment category
    sentiment_mapping = {0: 'very negative', 1: 'negative', 2: 'positive', 3: 'very positive', 4: 'neutral'}
    return sentiment_mapping[prediction]

# Step 4: Authenticate user based on sentiment
def authenticate_user(model, tokenizer, user_top_sentiments):
    user_id = input("Enter user ID: ")
    
    if user_id not in user_top_sentiments:
        print("User ID not found.")
        return
    
    # Get user's top 2 frequent sentiments
    top_2_sentiments = user_top_sentiments[user_id]
    print(f"User's top 2 frequent sentiments: {top_2_sentiments}")
    
    # Prompt for user input
    input_text = input("Enter a text prompt: ")
    
    # Predict sentiment of the input
    predicted_sentiment = predict_sentiment(model, tokenizer, input_text)
    print(f"Predicted sentiment: {predicted_sentiment}")
    
    # Authenticate if sentiment matches one of the top 2
    if predicted_sentiment in top_2_sentiments:
        print("Authentication successful.")
    else:
        print("Authentication failed.")

# Main function for loading, testing, and authentication
def main():
    # Load the dataset and calculate top sentiments (replace with actual dataset path)
    dataset_path = 'nlp_authentication_dataset.json'
    with open(dataset_path, 'r') as file:
        data = json.load(file)

    # Calculate the top 2 frequent sentiments for each user
    user_top_sentiments = calculate_frequent_sentiments(data)

    # Load the trained model and tokenizer
    model, tokenizer = load_model_and_tokenizer()

    # Authenticate user based on sentiment
    authenticate_user(model, tokenizer, user_top_sentiments)

if __name__ == "__main__":
    main()


User's top 2 frequent sentiments: ['negative', 'positive']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 249ms/step
Predicted sentiment: negative
Authentication successful.
